# Flipped-Class Notebook: Replicating Varian (2014), growth regressions

**Course:** EAIF: AI for Finance. **Activity:** read before class, replicate in class, reflect.

**Paper:** Hal R. Varian (2014), "Big Data: New Tricks for Econometrics", *Journal of Economic Perspectives* 28(2): 3-28.
- Article: https://www.aeaweb.org/articles?id=10.1257/jep.28.2.3
- Replication data (provenance, no download needed): OpenICPSR project 113925, https://www.openicpsr.org/openicpsr/project/113925

## What we replicate

Section 6.3 of the paper ("Economic example: growth regressions") compares four **variable-selection** methods on one small dataset: Bayesian model averaging (BMA, Ley and Steel 2009), Sala-i-Martin's CDF(0), the **lasso**, and spike-and-slab regression. The result is one table (Table 4 in the working-paper version) listing the ten predictors of economic growth that BMA ranks highest, with each method's verdict next to them. Varian's conclusion is that the four methods agree on the first four or five variables and then diverge, and that the dataset is too small to resolve what is "important" for growth.

The data are in `FLS-data.csv` (72 countries, 41 regressors, one outcome). We fit the lasso column of that table ourselves, then ask what a linear regression and a random forest say about the same data.

**Workflow**

1. Load the data (one line, from the course repository).
2. Inspect it and read the codebook.
3. Split into training and test rows.
4. Lasso: which variables survive, and in which order do they enter? Compare with the BMA ranking.
5. OLS baseline: what happens with 42 parameters and about 54 rows?
6. Random forest: does a non-linear learner rank the same variables?
7. Compare, then answer the reflection questions.

> You may use AI tools for coding help, but make sure you understand the code.


## 0) Setup
Run this cell first. Everything used below is preinstalled in Google Colab.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LassoCV, lasso_path
from sklearn.ensemble import RandomForestRegressor

# root_mean_squared_error exists from scikit-learn 1.4 on; older versions need the fallback
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    def root_mean_squared_error(y_true, y_pred):
        return np.sqrt(mean_squared_error(y_true, y_pred))

plt.rcParams['figure.dpi'] = 110
print('Ready. pandas', pd.__version__)


Ready. pandas 2.2.3


## 1) Get the data

**Option A (default):** the file is hosted in the course repository and loads with one line.

**Option B (fallback, only if Option A fails):** download `FLS-data.csv` from the OpenICPSR replication package (folder "Lasso") and upload it when the dialog appears.

In [2]:
DATA_URL = "https://raw.githubusercontent.com/umatter/EDFB/main/activity/FLS-data.csv"

try:
    df = pd.read_csv(DATA_URL)                       # Option A
    print(f"Loaded from the course repository: {df.shape[0]} rows x {df.shape[1]} columns")
except Exception as e:                               # Option B: manual upload in Colab
    print("Could not load from the repository:", e)
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded. Run the cell again and choose FLS-data.csv.")
    csv_name = list(uploaded.keys())[0]
    df = pd.read_csv(csv_name)
    print(f"Loaded uploaded file {csv_name}: {df.shape[0]} rows x {df.shape[1]} columns")


Loaded from the course repository: 72 rows x 42 columns


### Codebook

`FLS-data.csv` is the cross-country growth dataset of Fernandez, Ley and Steel (2001), "Model uncertainty in cross-country growth regressions", *Journal of Applied Econometrics* 16: 563-576. It goes back to Sala-i-Martin (1997) and was reused by Ley and Steel (2009) and by Varian (2014), Section 6.3.

- **Rows:** 72 countries.
- **`y`:** average growth rate of GDP per capita, 1960 to 1992 (percent per year).
- **41 regressors**, for example `GDPsh560` (log GDP per capita in 1960), `Confuncious` (fraction Confucian; the spelling is the original file's), `Life Exp` (life expectancy), `Equip Inv` (equipment investment share), `SubSahara` (dummy), `Muslim`, `Rule of Law`, `Yrs Open` (fraction of years 1950-1994 the economy was open), `Eco Org` (degree of capitalism), `Protestants`, and so on.
- **Column order matters:** the regressors appear in the order of the BMA posterior inclusion probability reported in Varian's table (Ley and Steel's ranking). The first ten columns after `y` are the ten rows of that table. We use this order as the "BMA ranking" below.


## 2) Inspect
Look at the structure, then set the target and the features. The target is `y`; every other column is a candidate regressor.

In [3]:
print(df.shape)
display(df.head())
display(df.describe().T.round(3))
print("Missing values in total:", int(df.isna().sum().sum()))

target_col = 'y'
feature_cols = [c for c in df.columns if c != target_col]
bma_rank = pd.Series(range(1, len(feature_cols) + 1), index=feature_cols, name='BMA_rank')
print(f"\nTarget: {target_col}; {len(feature_cols)} regressors. First ten by BMA ranking:")
print(list(bma_rank.index[:10]))


(72, 42)


,y,GDPsh560,Confuncious,Life Exp,Equip Inv,SubSahara,Muslim,Rule of Law,Yrs Open,Eco Org,...,Abs Lat,Work/Pop,High Enroll,Pop g,Brit Col,Outwar Or,Jewish,Rev & Coup,%Publ Edu,Area
0,1.3690,7.438972,0.0,47.3,0.0507,0,0.99,0.3333,0.000,0,...,3.6720,-1.330292,0.003,0.028417,0,1,0.005,0.123,0.0297,2.382
1,5.6195,6.284134,0.0,45.7,0.1309,1,0.00,0.8333,0.356,5,...,2.1536,-0.824395,0.000,0.033150,1,0,0.000,0.000,0.0300,0.582
2,1.2239,6.546785,0.0,43.4,0.0040,1,0.16,0.5000,0.156,5,...,1.0730,-0.716620,0.002,0.026343,0,1,0.000,0.080,0.0221,0.475
3,2.2322,6.968851,0.0,47.3,0.0498,1,0.02,0.3333,0.000,1,...,0.3684,-0.774140,0.005,0.027473,0,1,0.000,0.360,0.0311,0.342
4,-0.0553,5.517453,0.0,42.2,0.0060,1,0.42,0.5000,0.000,0,...,0.9007,-0.670017,0.001,0.022771,0,0,0.000,0.720,0.0095,1.222


,count,mean,std,min,25%,50%,75%,max
y,72.0,2.073,1.825,-2.077,0.939,2.031,2.873,6.618
GDPsh560,72.0,7.492,0.885,5.517,6.915,7.388,8.104,9.187
Confuncious,72.0,0.019,0.087,0.000,0.000,0.000,0.000,0.600
Life Exp,72.0,56.581,11.448,37.900,45.850,54.600,68.800,73.400
Equip Inv,72.0,0.044,0.035,0.003,0.015,0.036,0.065,0.148
SubSahara,72.0,0.208,0.409,0.000,0.000,0.000,0.000,1.000
Muslim,72.0,0.148,0.295,0.000,0.000,0.000,0.143,1.000
Rule of Law,72.0,0.551,0.335,0.000,0.167,0.500,1.000,1.000
Yrs Open,72.0,0.439,0.355,0.000,0.089,0.411,0.778,1.000
Eco Org,72.0,3.542,1.266,0.000,3.000,3.000,5.000,5.000


Missing values in total: 0

Target: y; 41 regressors. First ten by BMA ranking:
['GDPsh560', 'Confuncious', 'Life Exp', 'Equip Inv', 'SubSahara', 'Muslim', 'Rule of Law', 'Yrs Open', 'Eco Org', 'Protestants']


## 3) Train/test split
We hold out a quarter of the countries (18 of 72) to measure out-of-sample error. With so few rows, the test error of any model is noisy: a different `random_state` can change the ranking of models. Keep that in mind when you read the numbers.

In [ ]:
X = df[feature_cols].copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print(f"Training rows: {len(X_train)}, test rows: {len(X_test)}, regressors: {X.shape[1]}")

# Baseline every model must beat: predict the training mean
baseline_rmse = root_mean_squared_error(y_test, np.full(len(y_test), y_train.mean()))
print(f"Naive baseline (training mean) test RMSE: {baseline_rmse:.3f}")


## 4) Lasso: replicate the variable selection

Two things are computed here.

1. **Which variables survive.** `LassoCV` picks the penalty by 5-fold cross-validation on the training rows and returns the coefficients at that penalty. Regressors are standardised first, because the lasso penalises all coefficients equally and the columns are on very different scales.
2. **In which order variables enter the path.** As the penalty is relaxed, variables get non-zero coefficients one after another. That entry order is what Varian's "lasso" column reports (1 = first to enter). We compute the path on all 72 countries, as the paper did, and put the entry order next to the BMA ranking encoded in the column order.


In [6]:
# (1) LassoCV on the training rows, penalty chosen by 5-fold CV
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

lasso = LassoCV(cv=5, random_state=42, max_iter=20000).fit(X_train_s, y_train)
lasso_coef = pd.Series(lasso.coef_, index=feature_cols)
survivors = lasso_coef[lasso_coef != 0].sort_values(key=abs, ascending=False)
print(f"Penalty chosen by CV: alpha = {lasso.alpha_:.4f}")
print(f"Variables with non-zero coefficient: {len(survivors)} of {len(feature_cols)}")
print(survivors.round(3))

y_pred_lasso = lasso.predict(X_test_s)
lasso_rmse = root_mean_squared_error(y_test, y_pred_lasso)
lasso_mae = mean_absolute_error(y_test, y_pred_lasso)
lasso_r2 = r2_score(y_test, y_pred_lasso)
print(f"\nLasso test RMSE {lasso_rmse:.3f} (naive baseline {baseline_rmse:.3f}), MAE {lasso_mae:.3f}, R2 {lasso_r2:.3f}")


NameError: name 'X_train' is not defined

In [ ]:
# (2) Entry order along the lasso path, on all 72 countries (standardised)
X_all_s = StandardScaler().fit_transform(X)
# Explicit penalty grid from the largest penalty that zeroes every coefficient down to 1/1000 of it
alpha_max = np.max(np.abs(X_all_s.T @ (y.values - y.values.mean()))) / len(y)
alpha_grid = np.logspace(np.log10(alpha_max), np.log10(alpha_max * 1e-3), 300)
alphas, coefs, _ = lasso_path(X_all_s, y.values, alphas=alpha_grid)

# For each variable: the first grid point (largest penalty) at which its coefficient is non-zero
first_nonzero = [(np.flatnonzero(np.abs(coefs[j]) > 0)[0] if np.any(coefs[j] != 0) else np.inf)
                 for j in range(coefs.shape[0])]
entry = pd.DataFrame({'first_step': first_nonzero}, index=feature_cols)
entry['lasso_entry_order'] = entry['first_step'].rank(method='min').astype(int)
entry['BMA_rank'] = bma_rank

print("Varian's table rows (BMA ranking) with the lasso entry order from our path:")
print(entry.loc[bma_rank.index[:10], ['BMA_rank', 'lasso_entry_order']])
print("\nFirst ten variables to enter our lasso path:")
print(entry.sort_values('lasso_entry_order').head(10)[['lasso_entry_order', 'BMA_rank']])


**Look at the two tables.** Do the first variables to enter the lasso path match the top of the BMA ranking? Where do the two orders part ways? Varian's own lasso column shows dashes for initial GDP, life expectancy and rule of law: at his chosen penalty these three were not in the model even though BMA ranks them at the top. Note whether your path agrees.

## 5) Baseline: OLS with all 41 regressors

`statsmodels` OLS estimates 42 parameters (41 slopes plus an intercept) from the training rows. With about 54 rows that leaves very few degrees of freedom, and the fit is close to perfect in-sample by construction. The interesting part is the test error.

To compare effect sizes, never sort raw coefficients: `Pop g` is a fraction and `Area` is in millions of square kilometres, so their coefficients live on incomparable scales. We rank by **t-statistic** (coefficient divided by its standard error), which is unit-free, and also show standardised coefficients (effect of a one-standard-deviation change).

In [ ]:
X_train_c = sm.add_constant(X_train)
X_test_c = sm.add_constant(X_test)
ols_model = sm.OLS(y_train, X_train_c).fit()

y_pred_ols = ols_model.predict(X_test_c)
ols_rmse = root_mean_squared_error(y_test, y_pred_ols)
ols_mae = mean_absolute_error(y_test, y_pred_ols)
ols_r2 = r2_score(y_test, y_pred_ols)
print(f"OLS: {int(ols_model.df_model) + 1} parameters, {int(ols_model.df_resid)} residual degrees of freedom")
print(f"In-sample R2 {ols_model.rsquared:.3f}; test RMSE {ols_rmse:.3f} (naive baseline {baseline_rmse:.3f}), MAE {ols_mae:.3f}, R2 {ols_r2:.3f}")

# Rank effects by t-statistic, and show standardised coefficients (beta * sd(x))
ols_table = pd.DataFrame({
    'coef': ols_model.params.drop('const'),
    'std_coef': ols_model.params.drop('const') * X_train.std(),
    't': ols_model.tvalues.drop('const'),
    'p': ols_model.pvalues.drop('const'),
})
ols_table['BMA_rank'] = bma_rank
ols_table = ols_table.reindex(ols_table['t'].abs().sort_values(ascending=False).index)
print("\nTop 10 regressors by |t|:")
print(ols_table.head(10).round(3))


## 6) Extension: random forest

Varian does not fit a random forest to these data. We add one to ask what a non-linear learner makes of 72 rows. Tune a small grid by 5-fold CV on the training rows; the test rows are used once.

In [ ]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1)
param_grid = {'n_estimators': [300], 'max_depth': [None, 4, 8], 'min_samples_leaf': [1, 3, 5]}
gs = GridSearchCV(rf, param_grid, cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
gs.fit(X_train, y_train)
print('Best parameters:', gs.best_params_)
print(f"CV RMSE at best parameters: {-gs.best_score_:.3f}")

best_rf = gs.best_estimator_
y_pred_rf = best_rf.predict(X_test)
rf_rmse = root_mean_squared_error(y_test, y_pred_rf)
rf_mae = mean_absolute_error(y_test, y_pred_rf)
rf_r2 = r2_score(y_test, y_pred_rf)
print(f"Random forest test RMSE {rf_rmse:.3f} (naive baseline {baseline_rmse:.3f}), MAE {rf_mae:.3f}, R2 {rf_r2:.3f}")

rf_importance = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
rf_top = pd.DataFrame({'importance': rf_importance.head(10), 'BMA_rank': bma_rank[rf_importance.head(10).index]})
print("\nTop 10 by random-forest importance:")
print(rf_top.round(3))
rf_importance.head(10)[::-1].plot(kind='barh', title='Random forest: top 10 feature importances')
plt.xlabel('Importance'); plt.tight_layout(); plt.show()


## 7) Compare
One table of test errors next to the naive baseline, one plot of predictions, and the three variable rankings side by side.

In [ ]:
results = pd.DataFrame({
    'Naive mean': [baseline_rmse, mean_absolute_error(y_test, np.full(len(y_test), y_train.mean())), 0.0],
    'Lasso': [lasso_rmse, lasso_mae, lasso_r2],
    'OLS': [ols_rmse, ols_mae, ols_r2],
    'Random forest': [rf_rmse, rf_mae, rf_r2],
}, index=['RMSE', 'MAE', 'R2'])
print("Test-set performance (18 countries):")
print(results.round(3))

# Cross-validated RMSE on the training rows, for a second opinion that does not depend on one split
# Scaling inside each fold, so that the validation rows do not touch the scaler
cv_lasso = -cross_val_score(make_pipeline(StandardScaler(), LassoCV(cv=5, random_state=42, max_iter=20000)),
                            X_train, y_train, cv=5, scoring='neg_root_mean_squared_error').mean()
cv_ols = -cross_val_score(LinearRegression(), X_train, y_train, cv=5, scoring='neg_root_mean_squared_error').mean()
cv_rf = -gs.best_score_
print(f"\n5-fold CV RMSE on training rows: lasso {cv_lasso:.3f}, OLS {cv_ols:.3f}, random forest {cv_rf:.3f}")

plt.figure()
plt.scatter(y_test, y_pred_lasso, alpha=0.7, label='Lasso')
plt.scatter(y_test, y_pred_ols, alpha=0.7, label='OLS')
plt.scatter(y_test, y_pred_rf, alpha=0.7, label='Random forest')
lims = [float(y_test.min()), float(y_test.max())]
plt.plot(lims, lims, linestyle='--', color='grey')
plt.xlabel('Actual growth (% per year)'); plt.ylabel('Predicted'); plt.legend()
plt.title('Test countries: actual vs predicted'); plt.tight_layout(); plt.show()

rankings = pd.DataFrame({
    'BMA (column order)': bma_rank.index[:10],
    'Lasso entry order': entry.sort_values('lasso_entry_order').index[:10],
    'OLS |t|': ols_table.index[:10],
    'RF importance': rf_importance.index[:10],
}, index=range(1, 11))
print("\nTop ten variables under each method:")
print(rankings.to_string())


## 8) Reflection (write 3 to 5 sentences per question)

These are the same three questions you prepared before class. Answer them now with your numbers.

1. **The growth-regression table.** Which variables do the BMA ranking, your lasso path, OLS and the random forest agree on? Where do they diverge? Varian writes that the dataset appears too small to resolve what is important for growth: does your table support that?
2. **Prediction versus causation.** Section 3 of the paper is about getting good out-of-sample predictions (hence the train/test split and cross-validation). Section 8.1 asks a different question, with the police-and-crime example: more police are found where crime is high, which predicts crime well but says nothing about what adding police would do. Which of the two questions did we answer here? Could a policymaker use your model to decide where to invest?
3. **Model uncertainty.** Section 9 argues that the choice of model, not sampling noise, is often the larger source of uncertainty, and that averaging over models tends to predict better than picking one. Where did model uncertainty show up in your results (compare the three rankings and the three test errors), and what would averaging look like here?

**Deliverable (formative):** this notebook with your answers in text cells, submitted via Moodle by the end of class.


## Wrap-up

**1) What we replicated or learned**

We replicated Section 6.3 of Varian (2014): the Fernandez-Ley-Steel 72-country growth dataset, comparing BMA's published variable ranking against our own lasso (CV-selected penalty and full regularization path), an OLS baseline, and a random forest. We learned that out of 41 candidate growth predictors, only one -- equipment investment share -- is robust across every method; everything else, including BMA's top-ranked variable (initial 1960 GDP), looks very different depending on which method you ask.

**2) What the ML model added (or didn't add)**

The lasso and random forest added real robustness to overfitting: OLS fits 42 parameters on just 54 training rows (in-sample R² = 0.978) but its 5-fold CV error (4.893) is nearly 4x its single-split test error (1.259) -- a model that falls apart under resampling. Lasso (test R² = 0.651) and the random forest (R² = 0.560) both generalize far better, simply by penalizing complexity. What they didn't add: no new economic driver of growth. Every variable either method surfaces was already a candidate in the original 41 -- ML here is selecting and down-weighting among known candidates, not discovering anything new.

**3) One insight connecting ML to econometrics**

On this dataset, the ML methods are best understood as disciplined solutions to a problem econometricians already knew about -- too many regressors for too few observations -- rather than a fundamentally different kind of analysis. But the deeper econometric caution is untouched by any of this machinery: nothing here distinguishes correlation from causation (Varian's own police-and-crime example, Section 8.1), so even the most "robust" result -- equipment investment predicting growth -- cannot be read as "investing in equipment causes growth" without an identification strategy none of these models provide.